## Setup and imports

In [ ]:
EXP_NAME = "SPS_audit_1"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')

# Analysis

### Basics

In [ ]:
scorings = sps_audit.groupby('feature')[['sensitivity_scoring', 'fidelity_scoring']].first()

sps_audit = sps_audit.drop(['sensitivity_scoring', 'fidelity_scoring'], axis=1)

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['roc_auc']].count()
iteration_per_feature

### Counterfactual sensitivity

In [ ]:
# Counterfactual sensitivity
f, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=sps_audit, x="feature", y="cf_sensitivity", ax=ax)

plt.plot()


# Utility vs Fairness trade-off

In [ ]:
sps_audit.head()

In [ ]:
baseline_roc_auc = sbs_audit_baseline['roc_auc'].values[0]
baseline_ieco_mace = sbs_audit_baseline['ieco_mace'].values[0]

print(f"Baseline ROC AUC: {baseline_roc_auc}")
print(f"Baseline IECO MACE: {baseline_ieco_mace}")

In [ ]:
utility_vs_fairness = sps_audit.groupby('iteration')[['roc_auc', 'ieco_mace']].first().sort_values('ieco_mace')
x_desc_configs = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

pareto_frontier = []
current_max_utility = -1

print('--- Configurations on the Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['roc_auc'] > current_max_utility:
    pareto_frontier.append(solution)
    current_max_utility = solution['roc_auc']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs[idx]}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)


fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=utility_vs_fairness, x='roc_auc', y='ieco_mace', ax=ax)
ax.plot(baseline_roc_auc, baseline_ieco_mace, "or")
sns.lineplot(data=pareto_frontier_df, x='roc_auc', y='ieco_mace', color="green", marker='o')

plt.show()

In [ ]:
# feature_stats = sps_audit[sps_audit['bucket'] == 'x_desc']
feature_stats = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby(['feature'])[['ieco_mace', 'roc_auc']].mean()
feature_stats['ieco_gain'] = baseline_ieco_mace - feature_stats['ieco_mace']
feature_stats['auc_loss'] = baseline_roc_auc - feature_stats['roc_auc']
sns.scatterplot(data=feature_stats, x='ieco_gain', y='auc_loss', hue='feature')
plt.plot()

### $U_{desc}$ and $U_{corr}$ utility

In [ ]:
# Average Ucorr and Udesc utility

avg_utility = sps_audit.groupby('feature')[['cf_sensitivity','norm_f_desc', 'norm_f_corr']].agg('mean').sort_values(by='cf_sensitivity', ascending=False)
avg_utility['utility_delta'] = avg_utility['norm_f_desc'] - avg_utility['norm_f_corr']
avg_utility

# Bucket Assignment

What we infer from our metrics:

**Counterfactual Sensitivity (CS):** a high flip rate shows that the model has internalised a sociological pathway $S_{soc} \rightarrow X_{desc}$ where changing the sensitive attribute alters teh reconstructed feature

**Normalised $U_{desc}$ Utility:** Measures how much useful signal remains after the feature has been stripped of its relationship with $S$

**Utility Delta ($U_{desc} - U_{corr}$):** Measures the cost of $U_{desc}$ invariance by $S$. Since $U_{corr} is unconstrained, a large negative delta indicates the feature's predictive utility for the clinical outcome is inextricably tied to the sensitive attribute, and therefore has been stripped in $U_{desc}$

- **High CS (>= 0.2):**
  - **High $U_{desc}$ utility (>= 0.4):** The feature is heavily influenced by $S_{soc}$ and most of its clinical signal is preserved in $U_{desc}$ $\Rightarrow$ Assign to $X_{desc}$
  - **Low to Moderate $U_{desc}$ utility (< 0.4)** The feature is heavily influenced by $S_{soc}$, but cleaning it removes its clinical signal $\Rightarrow$ Assign to $X_{desc}$ to prioritise fairness, $X_{desc}$ to prioritise utility
    - Decision can be informed by the Utility Delta: a large negative hints at the feature's potential clinical signal to preserve
- **Low CS (< 0.2):**
  - **High $U_{corr}$ utility (>= 0.4) and large negative Utility Delta (>= -0.25):**  The feature's relationship with $S$ is likely biological ($S_{bio}$) $\Rightarrow$ Assign to $X_{corr}$
  - **Small negative Utility Delta (> -0.25):** The feature is insensitive to causal cosntraints on $S$ $\Rightarrow$ Assign to $X_{ind}$
  





In [ ]:
x_desc = []
x_corr = []
x_ind =[]
to_be_assigned = []

utility_threshold = 0.2

for feature, metrics in avg_utility.iterrows():
  if metrics["cf_sensitivity"] >= 0.2:
    if metrics["norm_f_desc"] >= utility_threshold:
      x_desc.append(feature)
    else:
      to_be_assigned.append({"feature": feature, "utility_delta": metrics['utility_delta']})
  else:
    if metrics["norm_f_corr"] >= utility_threshold and metrics['utility_delta'] <= -0.25:
      x_corr.append(feature)
    elif metrics['utility_delta'] > -0.25:
      x_ind.append(feature)
    else:
      to_be_assigned.append({"feature": feature, "utility_delta": metrics['utility_delta']})

print(f"X_desc: {x_desc}")
print(f"X_corr: {x_corr}")
print(f"X_ind: {x_ind}")
print("Remaining features to assign to X_desc or X_corr:")
print(pd.DataFrame(to_be_assigned).sort_values(by='utility_delta').to_markdown(index=False))
